# Omnichannel Journey Mapper

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Map customer journeys** with `LISTAGG()` for path creation
2. **Sequence touchpoints** using `ROW_NUMBER()`
3. **Calculate conversion paths** with window functions
4. **Identify high-performing journeys** with SQL analytics
5. **Build journey flow diagrams** (Sankey visualization data)

---

## What You'll Build

A **customer journey mapping system** that tracks omnichannel paths:
- Journey path construction (Email→Website→Event→Conversion)
- Conversion path analysis
- Channel sequencing patterns
- Touch optimization recommendations
- Sankey flow visualization data

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 14 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `LISTAGG()` - Concatenate journey steps into paths
- `ROW_NUMBER() OVER()` - Sequence events by time
- `LEAD()` - Identify next channel in journey
- `COUNT() OVER()` - Calculate total touches per journey
- Path analysis - Identify high-converting sequences

Let's begin!

---

## Paso 1: Configuración del Entorno

### Omnichannel Journey Mapping

Track how **Healthcare Providers (HCPs)** interact with your brand across:
- **Email campaigns** - Educational content, product updates
- **Website visits** - Disease information, clinical data
- **Medical conferences** - Booth visits, presentations
- **Rep visits** - In-person detailing

### Why Journey Mapping Matters

- Understand which touchpoint sequences lead to prescribing
- Optimize marketing spend on effective channels
- Personalize outreach based on journey stage
- Identify drop-off points in the funnel

In [ ]:
CREATE DATABASE IF NOT EXISTS JOURNEY_MAPPING_DB;
CREATE SCHEMA IF NOT EXISTS JOURNEY_MAPPING_DB.PUBLIC;
USE SCHEMA JOURNEY_MAPPING_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema, CURRENT_WAREHOUSE() as wh;

---

## Paso 2: Define Data Structure

### Tables

1. **`hcps`** - Healthcare provider profiles
2. **`touchpoints`** - All interactions (email, web, event, rep visit)
3. **`prescriptions`** - Prescription outcomes

### Journey Analysis

We'll track:
- **Touch sequence**: Order of interactions
- **Time between touches**: Days from one to next
- **Path patterns**: Common sequences (e.g., Email → Web → Rep → Rx)
- **Conversion rate**: % who prescribe after specific paths

In [ ]:
-- HCP profiles
CREATE OR REPLACE TABLE hcps (
    hcp_id VARCHAR(50) PRIMARY KEY,
    specialty VARCHAR(100),
    years_in_practice INTEGER,
    patient_volume VARCHAR(20),
    region VARCHAR(50),
    tier VARCHAR(20)  -- Tier 1 (high value), Tier 2, Tier 3
);

-- All touchpoints across channels
CREATE OR REPLACE TABLE touchpoints (
    touchpoint_id VARCHAR(50) PRIMARY KEY,
    hcp_id VARCHAR(50),
    touchpoint_date TIMESTAMP,
    channel VARCHAR(50),  -- Email, Website, Event, Rep_Visit
    touchpoint_type VARCHAR(100),  -- Specific action
    engagement_score INTEGER,  -- 1-10 engagement level
    content_topic VARCHAR(100)
);

-- Prescription outcomes
CREATE OR REPLACE TABLE prescriptions (
    prescription_id VARCHAR(50) PRIMARY KEY,
    hcp_id VARCHAR(50),
    prescription_date DATE,
    product_name VARCHAR(100),
    patient_count INTEGER
);

SELECT 'Tables created!' as status;

---

## Paso 3: Generar Datos Sintéticos Journey Data

Creating realistic HCP journey data:
- **500 HCPs** (cardiologists, ophthalmologists, oncologists)
- **5,000 touchpoints** across 4 channels
- **200 prescriptions** (40% conversion rate)
- Realistic timing and sequences

In [ ]:
-- Generar HCP profiles
INSERT INTO hcps
SELECT 
    'HCP' || LPAD(SEQ4(), 4, '0') as hcp_id,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Cardiology'
        WHEN 2 THEN 'Ophthalmology'
        WHEN 3 THEN 'Oncology'
        ELSE 'Primary Care'
    END as specialty,
    FLOOR(UNIFORM(2, 35, RANDOM())) as years_in_practice,
    CASE FLOOR(UNIFORM(1, 4, RANDOM()))
        WHEN 1 THEN 'High (200+ patients)'
        WHEN 2 THEN 'Medium (50-200)'
        ELSE 'Low (<50)'
    END as patient_volume,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Northeast'
        WHEN 2 THEN 'Southeast'
        WHEN 3 THEN 'Midwest'
        ELSE 'West'
    END as region,
    CASE 
        WHEN UNIFORM(0, 1, RANDOM()) > 0.7 THEN 'Tier 1'
        WHEN UNIFORM(0, 1, RANDOM()) > 0.5 THEN 'Tier 2'
        ELSE 'Tier 3'
    END as tier
FROM TABLE(GENERATOR(ROWCOUNT => 500));

-- Generar touchpoint journeys (10 touches per HCP on average)
INSERT INTO touchpoints
WITH hcp_base AS (
    SELECT hcp_id, tier, ROW_NUMBER() OVER (ORDER BY RANDOM()) as rn
    FROM hcps
),
touch_sequence AS (
    SELECT 
        hb.hcp_id,
        hb.tier,
        t.touch_num,
        -- First touch is usually Email or Website (awareness)
        CASE 
            WHEN t.touch_num = 1 THEN 
                CASE WHEN UNIFORM(0,1,RANDOM()) > 0.5 THEN 'Email' ELSE 'Website' END
            WHEN t.touch_num <= 3 THEN 
                CASE FLOOR(UNIFORM(1, 4, RANDOM()))
                    WHEN 1 THEN 'Email'
                    WHEN 2 THEN 'Website'
                    ELSE 'Event'
                END
            ELSE  -- Later touches include Rep visits
                CASE FLOOR(UNIFORM(1, 5, RANDOM()))
                    WHEN 1 THEN 'Email'
                    WHEN 2 THEN 'Website'
                    WHEN 3 THEN 'Event'
                    ELSE 'Rep_Visit'
                END
        END as channel,
        -- Time between touches: 5-30 days
        DATEADD('day', 
                (t.touch_num - 1) * FLOOR(UNIFORM(5, 30, RANDOM())), 
                DATEADD('month', -6, CURRENT_DATE())) as touch_date
    FROM hcp_base hb
    CROSS JOIN (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) as touch_num 
                FROM TABLE(GENERATOR(ROWCOUNT => 20))) t
    -- Tier 1 HCPs have more touches
    WHERE (hb.tier = 'Tier 1' AND t.touch_num <= FLOOR(UNIFORM(8, 15, RANDOM())))
       OR (hb.tier = 'Tier 2' AND t.touch_num <= FLOOR(UNIFORM(5, 10, RANDOM())))
       OR (hb.tier = 'Tier 3' AND t.touch_num <= FLOOR(UNIFORM(2, 6, RANDOM())))
)
SELECT 
    UUID_STRING() as touchpoint_id,
    hcp_id,
    touch_date as touchpoint_date,
    channel,
    -- Specific touchpoint types per channel
    CASE channel
        WHEN 'Email' THEN 
            CASE FLOOR(UNIFORM(1, 4, RANDOM()))
                WHEN 1 THEN 'Newsletter Open'
                WHEN 2 THEN 'Product Update Open'
                ELSE 'Clinical Study Open'
            END
        WHEN 'Website' THEN 
            CASE FLOOR(UNIFORM(1, 4, RANDOM()))
                WHEN 1 THEN 'Efficacy Data View'
                WHEN 2 THEN 'Safety Profile View'
                ELSE 'Dosing Guide View'
            END
        WHEN 'Event' THEN 
            CASE FLOOR(UNIFORM(1, 3, RANDOM()))
                WHEN 1 THEN 'Conference Booth Visit'
                ELSE 'Speaker Program Attendance'
            END
        ELSE  -- Rep_Visit
            CASE FLOOR(UNIFORM(1, 3, RANDOM()))
                WHEN 1 THEN 'Product Detailing'
                ELSE 'Sample Drop-off'
            END
    END as touchpoint_type,
    FLOOR(UNIFORM(1, 11, RANDOM())) as engagement_score,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Stroke Prevention'
        WHEN 2 THEN 'Cardiovascular Health'
        WHEN 3 THEN 'Vision Care'
        ELSE 'Cancer Treatment'
    END as content_topic
FROM touch_sequence;

-- Generar prescriptions (40% of HCPs prescribe)
INSERT INTO prescriptions
SELECT 
    UUID_STRING() as prescription_id,
    t.hcp_id,
    DATEADD('day', FLOOR(UNIFORM(7, 60, RANDOM())), MAX(t.touchpoint_date)::DATE) as prescription_date,
    CASE FLOOR(UNIFORM(1, 5, RANDOM()))
        WHEN 1 THEN 'Xarelto'
        WHEN 2 THEN 'Eylea'
        WHEN 3 THEN 'Adalat'
        ELSE 'Aspirin Cardio'
    END as product_name,
    FLOOR(UNIFORM(1, 15, RANDOM())) as patient_count
FROM touchpoints t
WHERE UNIFORM(0, 1, RANDOM()) > 0.6  -- 40% conversion rate
GROUP BY t.hcp_id;

-- Verify data
SELECT 
    (SELECT COUNT(*) FROM hcps) as hcps,
    (SELECT COUNT(*) FROM touchpoints) as touchpoints,
    (SELECT COUNT(*) FROM prescriptions) as prescriptions;

---

## Paso 4: Build Journey Sequences

### Qué Vamos a Crear

**Sequenced customer journeys** that show the path each HCP takes from first touch to conversion (or drop-off).

### Understanding Window Functions

Window functions perform calculations **across rows related to the current row** without collapsing them (like GROUP BY does). They're perfect for journey analysis!

```sql
ROW_NUMBER() OVER (PARTITION BY hcp_id ORDER BY touchpoint_date)
LEAD(channel) OVER (PARTITION BY hcp_id ORDER BY touchpoint_date)
LAG(touchpoint_date) OVER (PARTITION BY hcp_id ORDER BY touchpoint_date)
```

### Why Window Functions for Journey Mapping?

Window functions are ideal for analyzing sequences because:

- **Preserve All Rows**: Unlike GROUP BY, you see every touchpoint
- **Access Related Rows**: Look at previous/next touches in the journey
- **Partition by Customer**: Calculate separately for each HCP
- **Order Matters**: Chronological sequencing is built-in
- **No Self-Joins**: Cleaner, faster SQL than traditional approaches

### Key Window Functions Explained

#### **1. `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`**

Assigns a sequential number to each row within a partition:

```sql
ROW_NUMBER() OVER (PARTITION BY hcp_id ORDER BY touchpoint_date) as touch_sequence
```

**What It Does**:
- **`PARTITION BY hcp_id`**: Restart numbering for each HCP
- **`ORDER BY touchpoint_date`**: Number chronologically
- **Result**: 1, 2, 3, 4... for each HCP's journey

**Example Output**:
```
HCP_ID    | TOUCHPOINT_DATE | CHANNEL  | TOUCH_SEQUENCE
----------|-----------------|----------|----------------
HCP_001   | 2024-01-15      | Email    | 1
HCP_001   | 2024-02-10      | Website  | 2
HCP_001   | 2024-03-05      | Rep_Visit| 3
HCP_002   | 2024-01-20      | Website  | 1  ← Resets for new HCP
HCP_002   | 2024-02-15      | Email    | 2
```

#### **2. `LEAD(column) OVER (PARTITION BY ... ORDER BY ...)`**

Returns the **next row's value** in the ordered partition:

```sql
LEAD(channel) OVER (PARTITION BY hcp_id ORDER BY touchpoint_date) as next_channel
```

**What It Does**:
- Looks **ahead** to the next touchpoint
- Returns NULL for the last touchpoint in each journey
- Perfect for analyzing "What comes after Email?"

**Example Output**:
```
HCP_ID | TOUCHPOINT_DATE | CHANNEL   | NEXT_CHANNEL
-------|-----------------|-----------|-------------
HCP_001| 2024-01-15      | Email     | Website      ← Next touch
HCP_001| 2024-02-10      | Website   | Rep_Visit    ← Next touch
HCP_001| 2024-03-05      | Rep_Visit | NULL         ← Journey end
```

**Use Cases**:
- Build transition matrices (Email → Website: 35%)
- Identify drop-off points (last channel before stopping)
- Create Sankey flow diagrams

#### **3. `LAG(column) OVER (PARTITION BY ... ORDER BY ...)`**

Returns the **previous row's value** in the ordered partition:

```sql
LAG(touchpoint_date) OVER (PARTITION BY hcp_id ORDER BY touchpoint_date) as previous_touch_date
```

**What It Does**:
- Looks **backward** to the previous touchpoint
- Returns NULL for the first touchpoint in each journey
- Perfect for calculating time between touches

**Example Output**:
```
HCP_ID | TOUCHPOINT_DATE | PREVIOUS_TOUCH_DATE | DAYS_SINCE_LAST
-------|-----------------|---------------------|------------------
HCP_001| 2024-01-15      | NULL                | NULL             ← First touch
HCP_001| 2024-02-10      | 2024-01-15          | 26 days
HCP_001| 2024-03-05      | 2024-02-10          | 23 days
```

**Use Cases**:
- Calculate engagement velocity (days between touches)
- Identify stale leads (>60 days since last touch)
- Measure campaign cadence effectiveness

### The OVER Clause Anatomy

```sql
FUNCTION() OVER (
    PARTITION BY customer_id    -- Separate calculation per customer
    ORDER BY event_date         -- Define row sequence
    ROWS BETWEEN ... AND ...    -- Optional: define window frame
)
```

**`PARTITION BY`**: Divides data into groups (like GROUP BY), but doesn't collapse rows  
**`ORDER BY`**: Defines the sequence within each partition  
**`ROWS BETWEEN`**: (Optional) Limits which rows to include in calculation

### Real-World Journey Example

**HCP #12345's Journey**:

| Touch # | Date       | Channel   | Next Channel | Days Since Last | Path So Far           |
|---------|------------|-----------|--------------|------------------|-----------------------|
| 1       | 2024-01-15 | Email     | Website      | NULL             | Email                 |
| 2       | 2024-02-10 | Website   | Website      | 26               | Email → Website       |
| 3       | 2024-02-25 | Website   | Event        | 15               | Email → Website → Website |
| 4       | 2024-03-10 | Event     | Rep_Visit    | 14               | ... → Event           |
| 5       | 2024-03-25 | Rep_Visit | NULL         | 15               | ... → Rep_Visit (CONVERTED!) |

**Insights from Window Functions**:
- `ROW_NUMBER()`: Total touches = 5 (high engagement)
- `LEAD()`: Email → Website transition (effective)
- `LAG()`: Average 17.5 days between touches (optimal cadence)
- `COUNT() OVER()`: This is a high-converting journey pattern

### Why Not Use Self-Joins?

**Old Approach** (messy):
```sql
-- Finding next channel WITHOUT window functions
SELECT t1.channel, t2.channel as next_channel
FROM touchpoints t1
LEFT JOIN touchpoints t2 
  ON t1.hcp_id = t2.hcp_id 
  AND t2.touchpoint_date = (
    SELECT MIN(touchpoint_date) 
    FROM touchpoints 
    WHERE hcp_id = t1.hcp_id 
    AND touchpoint_date > t1.touchpoint_date
  )
```

**New Approach** (clean):
```sql
-- Same result with window functions
SELECT 
    channel,
    LEAD(channel) OVER (PARTITION BY hcp_id ORDER BY touchpoint_date) as next_channel
FROM touchpoints
```

**Benefits**: 10x faster, easier to read, no complex joins!

In [ ]:
-- Create sequenced journey paths
CREATE OR REPLACE TABLE journey_sequences AS
WITH ordered_touches AS (
    SELECT 
        t.hcp_id,
        t.touchpoint_id,
        t.touchpoint_date,
        t.channel,
        t.touchpoint_type,
        t.engagement_score,
        h.specialty,
        h.tier,
        -- Sequence number (1st touch, 2nd touch, etc.)
        ROW_NUMBER() OVER (PARTITION BY t.hcp_id ORDER BY t.touchpoint_date) as touch_sequence_num,
        -- Next channel in journey
        LEAD(t.channel) OVER (PARTITION BY t.hcp_id ORDER BY t.touchpoint_date) as next_channel,
        -- Days to next touch
        DATEDIFF('day', t.touchpoint_date, 
                 LEAD(t.touchpoint_date) OVER (PARTITION BY t.hcp_id ORDER BY t.touchpoint_date)) as days_to_next_touch,
        -- Did HCP eventually prescribe?
        CASE WHEN p.prescription_id IS NOT NULL THEN 'Converted' ELSE 'Not Converted' END as conversion_status,
        p.prescription_date
    FROM touchpoints t
    JOIN hcps h ON t.hcp_id = h.hcp_id
    LEFT JOIN prescriptions p ON t.hcp_id = p.hcp_id
)
SELECT * FROM ordered_touches;

-- View sample journeys
SELECT 
    hcp_id,
    touch_sequence_num,
    channel,
    touchpoint_type,
    days_to_next_touch,
    next_channel,
    conversion_status
FROM journey_sequences
WHERE hcp_id IN (SELECT hcp_id FROM journey_sequences LIMIT 5)
ORDER BY hcp_id, touch_sequence_num
LIMIT 25;

---

## Paso 5: Analyze Path Patterns

### Path Aggregation

Create **path strings** showing channel sequences:
- Example: "Email → Website → Event → Prescription"

### Path Performance

Which sequences have highest conversion rates?

In [ ]:
-- Aggregate full journey paths per HCP
CREATE OR REPLACE TABLE journey_paths AS
SELECT 
    hcp_id,
    specialty,
    tier,
    conversion_status,
    prescription_date,
    -- Build path string (e.g., "Email->Website->Rep_Visit")
    LISTAGG(channel, ' → ') WITHIN GROUP (ORDER BY touch_sequence_num) as full_path,
    COUNT(*) as total_touches,
    SUM(engagement_score) as total_engagement,
    AVG(days_to_next_touch) as avg_days_between_touches,
    MIN(touchpoint_date) as journey_start,
    MAX(touchpoint_date) as journey_end,
    DATEDIFF('day', journey_start, journey_end) as journey_duration_days
FROM journey_sequences
GROUP BY hcp_id, specialty, tier, conversion_status, prescription_date;

-- Create path performance table (for dashboard queries)
CREATE OR REPLACE TABLE path_performance AS
SELECT 
    full_path,
    COUNT(*) as hcp_count,
    COUNT(CASE WHEN conversion_status = 'Converted' THEN 1 END) as converted_count,
    ROUND(converted_count * 100.0 / hcp_count, 1) as conversion_rate,
    ROUND(AVG(total_touches), 1) as avg_touches,
    ROUND(AVG(total_engagement), 0) as avg_engagement,
    ROUND(AVG(journey_duration_days), 0) as avg_journey_days
FROM journey_paths
GROUP BY full_path
HAVING hcp_count >= 3  -- Only paths with 3+ HCPs
ORDER BY conversion_rate DESC, hcp_count DESC;

-- View top converting paths
SELECT 
    full_path,
    hcp_count,
    conversion_rate || '%' as conversion_pct,
    avg_touches,
    avg_journey_days || ' days' as journey_duration
FROM path_performance
WHERE conversion_rate > 0
ORDER BY conversion_rate DESC, hcp_count DESC
LIMIT 15;

---

## Paso 6: Channel Attribution Analysis

### First-Touch Attribution

Which channel **started** the most converted journeys?

### Last-Touch Attribution

Which channel was the **final touchpoint** before prescription?

### Multi-Touch Attribution

Credit all channels in the path

In [ ]:
-- Channel attribution analysis
CREATE OR REPLACE TABLE channel_attribution AS
WITH first_last_touch AS (
    SELECT 
        hcp_id,
        conversion_status,
        MIN(CASE WHEN touch_sequence_num = 1 THEN channel END) as first_touch_channel,
        MAX(CASE WHEN next_channel IS NULL THEN channel END) as last_touch_channel
    FROM journey_sequences
    GROUP BY hcp_id, conversion_status
),
channel_counts AS (
    SELECT 
        channel,
        COUNT(DISTINCT hcp_id) as total_hcps_touched,
        COUNT(DISTINCT CASE WHEN conversion_status = 'Converted' THEN hcp_id END) as converted_hcps
    FROM journey_sequences
    GROUP BY channel
),
first_touch_attr AS (
    SELECT 
        first_touch_channel as channel,
        COUNT(*) as first_touch_conversions
    FROM first_last_touch
    WHERE conversion_status = 'Converted'
    GROUP BY first_touch_channel
),
last_touch_attr AS (
    SELECT 
        last_touch_channel as channel,
        COUNT(*) as last_touch_conversions
    FROM first_last_touch
    WHERE conversion_status = 'Converted'
    GROUP BY last_touch_channel
)
SELECT 
    cc.channel,
    cc.total_hcps_touched,
    cc.converted_hcps,
    ROUND(cc.converted_hcps * 100.0 / cc.total_hcps_touched, 1) as conversion_rate,
    COALESCE(fta.first_touch_conversions, 0) as first_touch_conversions,
    COALESCE(lta.last_touch_conversions, 0) as last_touch_conversions
FROM channel_counts cc
LEFT JOIN first_touch_attr fta ON cc.channel = fta.channel
LEFT JOIN last_touch_attr lta ON cc.channel = lta.channel
ORDER BY cc.converted_hcps DESC;

-- View attribution results
SELECT 
    channel,
    total_hcps_touched as total_hcps,
    converted_hcps,
    conversion_rate || '%' as conv_rate,
    first_touch_conversions as first_touch,
    last_touch_conversions as last_touch
FROM channel_attribution
ORDER BY converted_hcps DESC;

---

## Paso 7: Prepare Sankey Diagram Data

### Sankey Flow Visualization

Shows how HCPs flow through channels:
- **Nodes**: Each channel (Email, Website, Event, Rep_Visit)
- **Links**: Flow from channel to next channel
- **Width**: Number of HCPs following that path

### Data Format

Sankey visualizers need:
- Source channel
- Target channel
- Value (HCP count)

In [ ]:
-- Generar Sankey flow data (channel transitions)
CREATE OR REPLACE TABLE sankey_flows AS
WITH channel_transitions AS (
    SELECT 
        channel as source_channel,
        next_channel as target_channel,
        conversion_status,
        COUNT(*) as transition_count
    FROM journey_sequences
    WHERE next_channel IS NOT NULL  -- Exclude last touch
    GROUP BY source_channel, target_channel, conversion_status
)
SELECT 
    source_channel,
    target_channel,
    conversion_status,
    transition_count,
    -- Sankey JSON format
    OBJECT_CONSTRUCT(
        'source', source_channel,
        'target', target_channel,
        'value', transition_count,
        'conversion_status', conversion_status
    ) as sankey_link
FROM channel_transitions
ORDER BY transition_count DESC;

-- View top flows
SELECT 
    source_channel || ' → ' || target_channel as flow,
    conversion_status,
    transition_count as hcp_count
FROM sankey_flows
ORDER BY transition_count DESC
LIMIT 20;

-- Export Sankey JSON array (for visualization tools)
SELECT 
    ARRAY_AGG(sankey_link) WITHIN GROUP (ORDER BY transition_count DESC) as sankey_data
FROM sankey_flows;

---

## Paso 8: Interactive Journey Dashboard

Visualize HCP journeys, conversion funnels, and channel attribution

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🗺️ Omnichannel Journey Mapper")
st.markdown("### HCP engagement paths and conversion analysis")

# Key metrics
col1, col2, col3, col4 = st.columns(4)

total_hcps = session.sql("SELECT COUNT(DISTINCT hcp_id) FROM journey_sequences").collect()[0][0]
converted = session.sql("SELECT COUNT(DISTINCT hcp_id) FROM journey_sequences WHERE conversion_status='Converted'").collect()[0][0]
avg_touches = float(session.sql("SELECT ROUND(AVG(total_touches),1) FROM journey_paths").collect()[0][0])
avg_days = float(session.sql("SELECT ROUND(AVG(journey_duration_days),0) FROM journey_paths WHERE conversion_status='Converted'").collect()[0][0])

with col1:
    st.metric("Total HCPs", total_hcps)
with col2:
    st.metric("Converted", f"{converted} ({round(converted*100/total_hcps,1)}%)")
with col3:
    st.metric("Avg Touches", avg_touches)
with col4:
    st.metric("Avg Journey", f"{int(avg_days)} days")

# Top converting paths
st.markdown("---")
st.subheader("🏆 Top Converting Journey Paths")

top_paths = session.sql("""
    SELECT full_path, hcp_count, conversion_rate, avg_touches, avg_journey_days
    FROM path_performance
    WHERE conversion_rate > 0
    ORDER BY conversion_rate DESC, hcp_count DESC
    LIMIT 10
""").to_pandas()

st.dataframe(top_paths, use_container_width=True, hide_index=True)

# Channel attribution
st.markdown("---")
st.subheader("📊 Channel Attribution Analysis")

col1, col2 = st.columns(2)

with col1:
    st.markdown("**First-Touch Attribution**")
    first_touch = session.sql("""
        SELECT channel, first_touch_conversions as conversions
        FROM channel_attribution
        WHERE first_touch_conversions > 0
        ORDER BY first_touch_conversions DESC
    """).to_pandas()
    st.bar_chart(first_touch.set_index('CHANNEL')['CONVERSIONS'])

with col2:
    st.markdown("**Last-Touch Attribution**")
    last_touch = session.sql("""
        SELECT channel, last_touch_conversions as conversions
        FROM channel_attribution
        WHERE last_touch_conversions > 0
        ORDER BY last_touch_conversions DESC
    """).to_pandas()
    st.bar_chart(last_touch.set_index('CHANNEL')['CONVERSIONS'])

# Channel performance
st.markdown("---")
st.subheader("🎯 Channel Performance")

channel_perf = session.sql("""
    SELECT channel, total_hcps_touched, converted_hcps, conversion_rate
    FROM channel_attribution
    ORDER BY converted_hcps DESC
""").to_pandas()

st.dataframe(channel_perf, use_container_width=True, hide_index=True)

# Sankey flow preview
st.markdown("---")
st.subheader("🌊 Channel Flow Analysis")

sankey_data = session.sql("""
    SELECT source_channel, target_channel, transition_count
    FROM sankey_flows
    WHERE conversion_status = 'Converted'
    ORDER BY transition_count DESC
    LIMIT 15
""").to_pandas()

st.markdown("**Top Channel Transitions (Converted HCPs)**")
st.dataframe(sankey_data, use_container_width=True, hide_index=True)

st.info("💡 Use these Sankey flows in visualization tools (Plotly, D3.js) to create interactive flow diagrams")

# Journey duration analysis
st.markdown("---")
st.subheader("⏱️ Journey Duration by Outcome")

col1, col2 = st.columns(2)

with col1:
    converted_duration = session.sql("""
        SELECT journey_duration_days as days
        FROM journey_paths
        WHERE conversion_status = 'Converted'
    """).to_pandas()
    st.markdown("**Converted Journeys**")
    st.line_chart(converted_duration['DAYS'].value_counts().sort_index())

with col2:
    not_converted_duration = session.sql("""
        SELECT journey_duration_days as days
        FROM journey_paths
        WHERE conversion_status = 'Not Converted'
    """).to_pandas()
    st.markdown("**Not Converted Journeys**")
    st.line_chart(not_converted_duration['DAYS'].value_counts().sort_index())

# Download
st.markdown("---")
csv = top_paths.to_csv(index=False)
st.download_button("📥 Download Journey Analysis", data=csv, file_name="journey_paths.csv", mime="text/csv")

---

## ✅ Tutorial Complete!

### What You've Learned

1. ✅ Tracked patient journeys across multiple channels
2. ✅ Analyzed sequential paths with window functions
3. ✅ Identified high-converting path patterns
4. ✅ Calculated channel attribution (first-touch, last-touch)
5. ✅ Prepared Sankey flow data for visualization
6. ✅ Measured journey duration and engagement

### Key SQL Techniques

- **`ROW_NUMBER()`**: Sequence touches chronologically
- **`LEAD()/LAG()`**: Track next/previous channel
- **`LISTAGG()`**: Build path strings
- **`OBJECT_CONSTRUCT()`**: Create JSON for Sankey
- **Window functions**: Partition by HCP for journey analysis

### Journey Insights

- **Multi-touch journeys**: Most conversions require 5-8 touches
- **Rep visits**: Often last touch before prescription
- **Email + Web**: Common awareness-building combination
- **Journey duration**: 30-90 days typical for conversion

### Applications

- **Marketing optimization**: Focus spend on effective paths
- **Personalization**: Tailor outreach based on journey stage
- **Funnel analysis**: Identify drop-off points
- **Channel mix**: Balance awareness vs. conversion channels

### Next Steps

- Add time-decay attribution models
- Incorporate campaign-level data
- Build predictive models for next-best-action
- Create real-time journey dashboards
- Integrate with marketing automation

### Resources

- [Window Functions](https://docs.snowflake.com/en/sql-reference/functions-analytic)
- [LISTAGG](https://docs.snowflake.com/en/sql-reference/functions/listagg)
- [OBJECT_CONSTRUCT](https://docs.snowflake.com/en/sql-reference/functions/object_construct)

---

**Congratulations!** You've built a complete omnichannel journey mapping system.

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `JOURNEY_MAPPING_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS JOURNEY_MAPPING_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;